Mount Drive & Imports

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define Paths (Adjust if your folder name is different)
BASE_PATH = '/content/drive/MyDrive/imdb_data'
METADATA_PATH = f'{BASE_PATH}/400k_Basic_genome_movielens_keywords.csv'
RATINGS_PATH = f'{BASE_PATH}/working_on_data/Users_Ratings/merged_users_ratings.csv'
OUTPUT_DIR = f'{BASE_PATH}/ekko_artifacts'

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Drive mounted and paths set.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and paths set.


Load & Clean Metadata

In [ ]:
# Load Metadata
print("Loading metadata...")
df_meta = pd.read_csv(METADATA_PATH)

# Clean Missing Values
# We replace NaN with empty strings so they don't break the text blob
fill_cols = ['description', 'keywords', 'actors', 'directors']
for col in fill_cols:
    df_meta[col] = df_meta[col].fillna('')

# Create the "Llama Text Blob"
# Format: "Title: X. Genres: Y. Overview: Z. Cast: A. Directors: B. Keywords: C."
print("Creating Text Blobs...")

def create_blob(row):
    return (
        f"Title: {row['primaryTitle']}. "
        f"Genres: {row['genres']}. "
        f"Overview: {row['description']}. "
        f"Cast: {row['actors']}. "
        f"Directors: {row['directors']}. "
        f"Keywords: {row['keywords']}."
    )

df_meta['text_blob'] = df_meta.apply(create_blob, axis=1)

# Keep only essential columns for the App (Compression)
# We need 'tconst' for ID, 'text_blob' for AI, and others for UI display/filtering
keep_cols = [
    'tconst', 'primaryTitle', 'startYear', 'titleType',
    'genres', 'averageRating', 'numVotes', 'runtimeMinutes',
    'text_blob', 'actors' # Kept actors raw for the "Search by Actor" feature
]
df_final = df_meta[keep_cols]

print(f" Metadata Cleaned. Shape: {df_final.shape}")
print("Sample Blob:\n", df_final['text_blob'].iloc[0][:200] + "...")

Loading metadata...
Creating Text Blobs...
 Metadata Cleaned. Shape: (428167, 10)
Sample Blob:
 Title: A.M. Joy. Genres: News. Overview: Breaking news and discussion about politics and 2016 race.. Cast: Joy-Ann Reid, Jason Johnson, Jonathan Capehart, Raul A. Reyes, Malcolm Nance, Naveed Jamali, ...


Extract the "VIP List" (Top 2000)

In [ ]:
# Sort by popularity (numVotes) and take top 2000
vip_df = df_final.sort_values(by='numVotes', ascending=False).head(2000)

# Save just the ID and Text Blob for the Llama generator script
vip_list = vip_df[['tconst', 'primaryTitle', 'text_blob']].copy()

print(f" VIP List extracted: {len(vip_list)} items.")

 VIP List extracted: 2000 items.


Prepare Classification Labels (The Target)

This converts your 1-10 stars into Binary Labels for the LightGBM classifier.

Like (1): Rating $\ge$ 7

Dislike (0): Rating $\le$ 5

(We drop ratings of 6 to reduce noise).

In [ ]:
print("Loading ratings (this might take a minute)...")
# Load only necessary columns to save RAM
df_ratings = pd.read_csv(RATINGS_PATH, usecols=['userID', 'tconst', 'rating'])

# Filter out the "ambiguous" ratings (Rating = 6)
df_labeled = df_ratings[ (df_ratings['rating'] >= 7) | (df_ratings['rating'] <= 5) ].copy()

# Create Binary Target
# 1 if Rating >= 7, else 0
df_labeled['target'] = df_labeled['rating'].apply(lambda x: 1 if x >= 7 else 0)

print(f"Original Ratings: {len(df_ratings)}")
print(f"Labeled Ratings (Binary): {len(df_labeled)}")
print(f"Class Balance:\n{df_labeled['target'].value_counts(normalize=True)}")

Loading ratings (this might take a minute)...


/tmp/ipython-input-2378816397.py:3: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_ratings = pd.read_csv(RATINGS_PATH, usecols=['userID', 'tconst', 'rating'])


Original Ratings: 36647302
Labeled Ratings (Binary): 30203062
Class Balance:
target
1    0.764731
0    0.235269
Name: proportion, dtype: float64


Save Artifacts

In [ ]:
# Fix Types & Save Artifacts

print("Fixing data types...")

# Force userID to be integers (handles string issues)
# errors='coerce' turns non-numbers into NaN, then we drop them just in case
df_labeled['userID'] = pd.to_numeric(df_labeled['userID'], errors='coerce')
df_labeled = df_labeled.dropna(subset=['userID'])
df_labeled['userID'] = df_labeled['userID'].astype(int)

# Ensure tconst is string (it usually is, but let's be safe)
df_labeled['tconst'] = df_labeled['tconst'].astype(str)

print("Saving artifacts to Drive...")

# Save Metadata
df_final.to_parquet(f'{OUTPUT_DIR}/metadata_enriched.parquet', index=False)

# Save VIP List
vip_list.to_csv(f'{OUTPUT_DIR}/vip_list.csv', index=False)

# Save Ratings
df_labeled.to_parquet(f'{OUTPUT_DIR}/ratings_labeled.parquet', index=False)

print(f" DONE! All files saved to: {OUTPUT_DIR}")

Fixing data types...


/tmp/ipython-input-2089461931.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labeled['userID'] = df_labeled['userID'].astype(int)
/tmp/ipython-input-2089461931.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_labeled['tconst'] = df_labeled['tconst'].astype(str)


Saving artifacts to Drive...
 DONE! All files saved to: /content/drive/MyDrive/imdb_data/ekko_artifacts
